# 读取csv

In [1]:
import pandas as pd

# 1) 读取 CSV
file_path = "../data/4.数据处理/transport_result_1_4.csv"  # 改成你的文件路径
df = pd.read_csv(file_path)

# 2) 显示前 5 行
df.head(5)

,row_id,案号,裁判日期,当事人,case_time,province,city,district,specific_place,vehicle,death
0,0,（2016）浙0109民初16975号,2001-01-01,"陈波波,王雷,华安财产保险股份有限公司安徽分公司宿州中心支公司灵璧营销服务部",2016年06月30日09时,浙江省,杭州市,萧山区,横一线建一村路口,0,0.0
1,1,（2017）浙0109民初3106号,2001-01-01,"施建康,姚路兴,中国人民财产保险股份有限公司杭州市分公司",2013年12月28日15时,浙江省,杭州市,萧山区,瓜沥镇坎山工农村新农村地方,1,0.0
2,2,（2017）浙0109民初2322号,2001-01-01,周水娥；谭继明；信达财产保险股份有限公司浙江分公司,2014年10月26日13时20分,浙江省,杭州市,萧山区,临浦镇临浦电厂江塘处,1,0.0
3,3,（2017）浙0109民初2852号,2001-01-01,胡生羊；宋建国；杭州展铭钢铁有限公司；中国人民财产保险股份有限公司杭州市分公司,2015年12月5日11时40分,浙江省,杭州市,萧山区,青六线杨家桥南300米处,1,0.0
4,4,（2011）海法民一初字第42号,2001-06-22,吴略；中国太平洋财产保险股份有限公司深圳分公司（下称深圳太平洋保险公司）；深圳市中原集装箱运...,2001年06月22日00时,广东省,海丰县,海丰县,NaN,NaN,0.0


In [3]:
print(f"总行数：{len(df)} 行")

总行数：384062 行


# 去除“具体地点”中的 空值/xX/2个字及以下

In [5]:
# 指定要处理的列名
col = "specific_place"

# 去除该列为空（NaN）以及空字符串/全空格的行
df_clean_nall = df.copy()

# 先把纯空格变成 NaN（可选但很实用）
df_clean_nall[col] = df_clean_nall[col].astype("string").str.strip()
df_clean_nall.loc[df_clean_nall[col] == "", col] = pd.NA

# 再丢掉该列为 NA 的行
df_clean_nall = df_clean_nall.dropna(subset=[col])
print(f"去除空值后的行数：{len(df_clean_nall)} 行")

去除空值后的行数：304400 行


In [7]:
# 去掉包含 X 或 x 的行（只要字符串里出现就剔除）---
mask_has_x = df_clean_nall[col].str.contains(r"[x × X * 某]", na=False)
df_clean_xX = df_clean_nall.loc[~mask_has_x].copy()
print(f"去除包含 x/X 后的行数：{len(df_clean_xX)} 行")

去除包含 x/X 后的行数：294374 行


In [9]:
# 去掉“汉字2个及以下”的行（只统计汉字数量）---
# 统计每行中的汉字个数：[\u4e00-\u9fff] 覆盖常用汉字区间
han_count = df_clean_xX[col].str.findall(r"[\u4e00-\u9fff]").str.len()

mask_han_le2 = han_count <= 2
df_clean_text = df_clean_xX.loc[~mask_han_le2].copy()
print(f"去除汉字<=2 后的行数：{len(df_clean_text)} 行")

去除汉字<=2 后的行数：286667 行


In [11]:
# 删除 rowid 列（如果不存在也不报错）
df_clean_id = df_clean_text.drop(columns=["row_id"], errors="ignore")
print(f"去除row_id 后的行数：{len(df_clean_text)} 行")

去除row_id 后的行数：286667 行


# 查看结果

In [13]:
df_clean_id.head(5)

,案号,裁判日期,当事人,case_time,province,city,district,specific_place,vehicle,death
0,（2016）浙0109民初16975号,2001-01-01,"陈波波,王雷,华安财产保险股份有限公司安徽分公司宿州中心支公司灵璧营销服务部",2016年06月30日09时,浙江省,杭州市,萧山区,横一线建一村路口,0,0.0
1,（2017）浙0109民初3106号,2001-01-01,"施建康,姚路兴,中国人民财产保险股份有限公司杭州市分公司",2013年12月28日15时,浙江省,杭州市,萧山区,瓜沥镇坎山工农村新农村地方,1,0.0
2,（2017）浙0109民初2322号,2001-01-01,周水娥；谭继明；信达财产保险股份有限公司浙江分公司,2014年10月26日13时20分,浙江省,杭州市,萧山区,临浦镇临浦电厂江塘处,1,0.0
3,（2017）浙0109民初2852号,2001-01-01,胡生羊；宋建国；杭州展铭钢铁有限公司；中国人民财产保险股份有限公司杭州市分公司,2015年12月5日11时40分,浙江省,杭州市,萧山区,青六线杨家桥南300米处,1,0.0
5,（2002）绍刑初字第75号,2002-02-05,朱某,2001年11月7日06时50分,浙江省,绍兴市,柯桥区,104国道绍兴县柯桥地段,2,1.0


# 保存结果

In [15]:
output_path = "../data/5.清洗结果/clean_transport_result_1_4.csv"

df_clean_id.to_csv(output_path, index=False, encoding="utf-8-sig")